# NB1 — RAG Pipeline & Métriques
**Projet :** Assistant de Prédiction des Catastrophes Climatiques  
**Équipe :** Diego MERCHAN, Camille KOENIG, P3 Jayson NGUYEN PHANG, P4 Xia BIZOT

**Objectif :** Évaluer et comparer les retrievers RAG sur le corpus climatique

## Plan du notebook
1. Hypothèse
2. Configuration & imports
3. Chargement du vector store
4. Définition des 10 questions de test
5. Retriever Dense (MMR) — résultats et citations
6. Retriever Hybride (BM25 + Dense)
7. Métriques comparatives : recall@k, MRR
8. Test d'idempotence du vector store
9. Introduction RAGAS
10. Tableau récapitulatif et conclusion

## 1. Hypothèse
**Hypothèse :** "Le retriever hybride (BM25+Dense) retourne des documents plus pertinents que le retriever dense seul (MMR), en particulier sur les termes spécifiques comme CELEX, IPCC, ou les noms de rapports."

In [6]:
## 2. Configuration & imports

import sys
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
np.random.seed(SEED)

# Chemins relatifs — fonctionne en notebook et en script
ROOT = Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(f"Racine du projet : {ROOT}")
print(f"SEED = {SEED}")

Racine du projet : c:\Users\dmerchan\catastrophes-climatiques-rag
SEED = 42


## 3. Chargement du vector store
On charge le vector store FAISS déjà généré par `embeddings.py`.  
Si le fichier `faiss_index/` est absent du disque, on repart de zéro : les chunks du corpus `data/raw/` sont rechargés, découpés, puis vectorisés avant d'être sauvegardés sur disque pour les prochaines exécutions.

In [17]:
import faiss
import pickle
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

# Chargement direct sans sentence_transformers
FAISS_PATH = str(ROOT / "faiss_store")

# FakeEmbeddings pour contourner l'import de torch
# On n'a pas besoin de réencoder — le vecteur store est déjà sur disque
embeddings = FakeEmbeddings(size=384)
vector_store = FAISS.load_local(
    FAISS_PATH, 
    embeddings, 
    allow_dangerous_deserialization=True
)
print(f"Nombre de vecteurs : {vector_store.index.ntotal}")

Nombre de vecteurs : 1889


## 4. Définition des 10 questions de test
Le jeu de test est divisé en deux groupes pour valider l'hypothèse :
- **Questions générales (1–5)** : portent sur des phénomènes climatiques sans nommer de rapport précis. Le retriever dense seul devrait s'en sortir correctement.
- **Questions spécifiques (6–10)** : citent explicitement des noms de rapports (IPCC, CELEX, Copernicus, EMDAT, NOAA). Ces termes rares sont peu représentés dans l'espace vectoriel dense ; c'est là que BM25 apporte sa valeur ajoutée.

In [10]:
QUESTIONS = [
    # Questions générales
    "Quelles régions sont les plus vulnérables aux inondations selon le GIEC ?",
    "Quels sont les impacts du changement climatique sur les sécheresses ?",
    "Comment les ouragans sont-ils liés au réchauffement climatique ?",
    "Quels sont les seuils critiques de précipitations pour les inondations ?",
    "Quelles sont les prévisions d'élévation du niveau de la mer ?",
    # Questions avec termes spécifiques (pour tester l'hypothèse)
    "Que dit le rapport IPCC AR6 sur les événements extrêmes ?",
    "Quelles sont les recommandations du rapport CELEX sur les inondations ?",
    "Que dit le rapport Copernicus sur la température en Europe en 2023 ?",
    "Quels événements climatiques sont documentés dans le rapport EMDAT ?",
    "Que dit le rapport NOAA sur la saison des ouragans 2023 ?"
]
print(f"{len(QUESTIONS)} questions de test définies")

10 questions de test définies


## 5. Retriever Dense (MMR) — résultats et citations
Le retriever MMR (**Maximum Marginal Relevance**) sélectionne les documents en équilibrant similarité avec la requête et diversité entre eux, évitant ainsi les réponses redondantes.  
Paramètres retenus : `k=4` documents retournés, `fetch_k=10` candidats initiaux.  
**Avantage :** diversité des résultats.  
**Limite :** basé uniquement sur la similarité vectorielle — peut rater les termes rares ou les noms propres peu fréquents dans le corpus.

In [11]:
retriever_dense = creer_retriever(vector_store)

resultats_dense = []
for question in QUESTIONS:
    debut = time.time()
    resultat = interroger_rag(retriever_dense, question)
    duree = time.time() - debut

    docs = resultat["documents"]
    sources = [os.path.basename(d.metadata.get("source", "?")) for d in docs]

    resultats_dense.append({
        "question": question[:60] + "...",
        "nb_docs": len(docs),
        "sources": sources,
        "duree_ms": round(duree * 1000, 1)
    })

df_dense = pd.DataFrame(resultats_dense)
print("=== Résultats Retriever Dense (MMR) ===")
print(df_dense.to_string(index=False))

4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
4 document(s) trouvé(s) pour la question.
=== Résultats Retriever Dense (MMR) ===
                                                       question  nb_docs                                                                                                                                                                                                                                      sources  duree_ms
Quelles régions sont les plus vulnérables aux inondations se...        4                                                                                                                                       [Forest_Fires

### Exemple de réponse avec citations
On affiche le contexte complet retourné pour la première question afin de vérifier la qualité des extraits et le format des citations `[Source: ..., Page: ...]`.

In [12]:
question_exemple = QUESTIONS[0]
resultat = interroger_rag(retriever_dense, question_exemple)
print(f"Question : {question_exemple}\n")
print("=== Contexte récupéré avec citations ===")
print(resultat["contexte"])

4 document(s) trouvé(s) pour la question.
Question : Quelles régions sont les plus vulnérables aux inondations selon le GIEC ?

=== Contexte récupéré avec citations ===
[Source: Forest_Fires_2024.pdf, Page: 102]
101 
total burnt area around 12 thousand hectares. 
The combined burnt area of these two regions 
represents 98% of the total burnt area (Table 
34). Approximately 53% of all fires in 2024 
occurred in August and September (Table 35). 
September alone accounted for the vast 
majority of the burned area, i.e., around 92% of 
the total area affected that year. 
Table 34. Number of fires and burnt area in Mainland Portugal (NUTS 2). 
PT1 - NUTS 2 Region Number of 
fires 
Burnt area (ha) 
Forest and 
other wooded 
land 
Shrublands Agricultural 
land Total 
Norte 3 560 33 276 28 724 3 457 65 457 
Centro 1 077 46 760 16 688 5 082 68 530 
Oeste e Vale do Tejo 608 158 89 53 301 
Grande Lisboa 98 23 81 21 126 
Península de Setúbal 131 808 35 10 853 
Alentejo 606 971 450 918 2 340 
Algar

## 6. Retriever Hybride (BM25 + Dense)
**BM25** (Best Match 25) est une méthode de recherche par mots-clés basée sur TF-IDF amélioré : elle pénalise les termes trop fréquents et récompense les occurrences dans les documents courts.  
En combinant BM25 (exact match) et le retriever dense (sémantique) via un `EnsembleRetriever` avec des poids égaux (`0.5 / 0.5`), on obtient un retriever hybride capable de retrouver à la fois le sens et les termes exacts — ce qui est crucial pour des noms comme *CELEX*, *EMDAT* ou *Copernicus*.

In [22]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
# Chargement des chunks pour alimenter BM25 (index inversé sur le texte brut)
chunks = charger_et_decouper("data/raw")

# Retriever BM25 pur
retriever_bm25 = BM25Retriever.from_documents(chunks)
retriever_bm25.k = 4

# Fusion : 50 % BM25 + 50 % Dense (MMR)
retriever_hybride = EnsembleRetriever(
    retrievers=[retriever_bm25, retriever_dense],
    weights=[0.5, 0.5]
)
print("Retriever hybride (BM25 + Dense) créé avec weights=[0.5, 0.5]")

Chargement des documents depuis 'data/raw'...
  Chargé : 2023_Atlantic_Hurricane_Season.pdf (18 pages)
  Chargé : 2023_EMDAT_report.pdf (8 pages)
  Chargé : 2024_State_of_Global_Water_Resources.pdf (34 pages)
  Chargé : CELEX_52021DC0970_EN_TXT.pdf (18 pages)
  Chargé : European_state_climate_2023.pdf (28 pages)
  Chargé : Forest_Fires_2024.pdf (277 pages)
  Chargé : Global_Assessment_Report_Disaster_Risk_Reduction_2022.pdf (256 pages)
  Chargé : IPCC_AR6_SYR_SPM.pdf (42 pages)
  Chargé : IPCC_AR6_WGIII_SummaryForPolicymakers.pdf (56 pages)
  Chargé : IPCC_AR6_WGII_SummaryForPolicymakers.pdf (34 pages)

Total : 10 fichiers chargés, 771 pages au total.

Découpage des documents en chunks...
Découpage terminé : 1889 chunks créés.
Retriever hybride (BM25 + Dense) créé avec weights=[0.5, 0.5]


In [23]:
resultats_hybride = []
for question in QUESTIONS:
    debut = time.time()
    docs = retriever_hybride.invoke(question)
    duree = time.time() - debut

    sources = [os.path.basename(d.metadata.get("source", "?")) for d in docs]

    resultats_hybride.append({
        "question": question[:60] + "...",
        "nb_docs": len(docs),
        "sources": sources,
        "duree_ms": round(duree * 1000, 1)
    })

df_hybride = pd.DataFrame(resultats_hybride)
print("=== Résultats Retriever Hybride (BM25 + Dense) ===")
print(df_hybride.to_string(index=False))

=== Résultats Retriever Hybride (BM25 + Dense) ===
                                                       question  nb_docs                                                                                                                                                                                                                                                                                                                                                                 sources  duree_ms
Quelles régions sont les plus vulnérables aux inondations se...        8           [2023_EMDAT_report.pdf, Global_Assessment_Report_Disaster_Risk_Reduction_2022.pdf, 2024_State_of_Global_Water_Resources.pdf, IPCC_AR6_WGII_SummaryForPolicymakers.pdf, 2024_State_of_Global_Water_Resources.pdf, Global_Assessment_Report_Disaster_Risk_Reduction_2022.pdf, Forest_Fires_2024.pdf, Global_Assessment_Report_Disaster_Risk_Reduction_2022.pdf]      66.0
Quels sont les impacts du changement climatique sur les séch...

## 7. Métriques comparatives : recall@k et MRR
Deux métriques classiques en évaluation de systèmes de recherche d'information :
- **recall@k** : proportion de documents pertinents présents parmi les `k` premiers résultats retournés. Un recall@4 de 1.0 signifie que tous les documents attendus figurent dans les 4 premiers.
- **MRR** (Mean Reciprocal Rank) : inverse du rang du premier document pertinent trouvé. MRR = 1.0 si le premier résultat est pertinent, 0.5 si c'est le deuxième, etc.

Le *ground truth* ci-dessous est simplifié : on liste les fichiers sources attendus par question, basés sur la correspondance thématique avec le corpus disponible.

In [24]:
# Ground truth simplifié : sources attendues par question
GROUND_TRUTH = {
    QUESTIONS[0]: ["IPCC_AR6_WGII_SummaryForPolicymakers.pdf"],
    QUESTIONS[1]: ["IPCC_AR6_SYR_SPM.pdf", "2024_State_of_Global_Water_Resources.pdf"],
    QUESTIONS[2]: ["2023_Atlantic_Hurricane_Season.pdf"],
    QUESTIONS[3]: ["IPCC_AR6_WGII_SummaryForPolicymakers.pdf", "CELEX_52021DC0970_EN_TXT.pdf"],
    QUESTIONS[4]: ["IPCC_AR6_SYR_SPM.pdf"],
    QUESTIONS[5]: ["IPCC_AR6_SYR_SPM.pdf", "IPCC_AR6_WGII_SummaryForPolicymakers.pdf"],
    QUESTIONS[6]: ["CELEX_52021DC0970_EN_TXT.pdf"],
    QUESTIONS[7]: ["European_state_climate_2023.pdf"],
    QUESTIONS[8]: ["2023_EMDAT_report.pdf"],
    QUESTIONS[9]: ["2023_Atlantic_Hurricane_Season.pdf"],
}


def calculer_recall_at_k(sources_recuperees: list, sources_pertinentes: list, k: int = 4) -> float:
    """Proportion de sources pertinentes présentes dans les k premiers résultats."""
    sources_k = sources_recuperees[:k]
    pertinents_trouves = sum(1 for s in sources_k if s in sources_pertinentes)
    return pertinents_trouves / len(sources_pertinentes) if sources_pertinentes else 0.0


def calculer_mrr(sources_recuperees: list, sources_pertinentes: list) -> float:
    """Inverse du rang du premier document pertinent trouvé."""
    for i, source in enumerate(sources_recuperees):
        if source in sources_pertinentes:
            return 1.0 / (i + 1)
    return 0.0


metriques = []
for i, question in enumerate(QUESTIONS):
    pertinents = GROUND_TRUTH[question]

    sources_dense   = resultats_dense[i]["sources"]
    sources_hybride = resultats_hybride[i]["sources"]

    metriques.append({
        "question":          question[:50] + "...",
        "recall@4_dense":    round(calculer_recall_at_k(sources_dense,   pertinents), 2),
        "recall@4_hybride":  round(calculer_recall_at_k(sources_hybride, pertinents), 2),
        "MRR_dense":         round(calculer_mrr(sources_dense,   pertinents), 2),
        "MRR_hybride":       round(calculer_mrr(sources_hybride, pertinents), 2),
    })

df_metriques = pd.DataFrame(metriques)
print(df_metriques.to_string(index=False))
print(f"\nMoyenne recall@4 Dense   : {df_metriques['recall@4_dense'].mean():.2f}")
print(f"Moyenne recall@4 Hybride : {df_metriques['recall@4_hybride'].mean():.2f}")
print(f"Moyenne MRR Dense        : {df_metriques['MRR_dense'].mean():.2f}")
print(f"Moyenne MRR Hybride      : {df_metriques['MRR_hybride'].mean():.2f}")

                                             question  recall@4_dense  recall@4_hybride  MRR_dense  MRR_hybride
Quelles régions sont les plus vulnérables aux inon...             0.0               1.0       0.00         0.25
Quels sont les impacts du changement climatique su...             0.0               1.0       0.00         1.00
Comment les ouragans sont-ils liés au réchauffemen...             0.0               0.0       0.00         0.14
Quels sont les seuils critiques de précipitations ...             0.5               0.0       0.50         0.00
Quelles sont les prévisions d'élévation du niveau ...             1.0               0.0       1.00         0.00
Que dit le rapport IPCC AR6 sur les événements ext...             0.0               0.5       0.00         1.00
Quelles sont les recommandations du rapport CELEX ...             0.0               0.0       0.00         0.00
Que dit le rapport Copernicus sur la température e...             1.0               1.0       0.25      

## 8. Test d'idempotence du vector store
Un vector store FAISS est **idempotent** si deux chargements successifs depuis le même fichier produisent exactement le même nombre de vecteurs et les mêmes résultats de recherche.  
Ce test garantit que la sérialisation/désérialisation FAISS est stable et que les résultats du pipeline ne varient pas selon l'ordre d'exécution des cellules.

In [27]:
from langchain_community.embeddings import FakeEmbeddings
from langchain_community.vectorstores import FAISS

FAISS_PATH = str(ROOT / "faiss_store")

vs1 = FAISS.load_local(FAISS_PATH, FakeEmbeddings(size=384), allow_dangerous_deserialization=True)
vs2 = FAISS.load_local(FAISS_PATH, FakeEmbeddings(size=384), allow_dangerous_deserialization=True)

assert vs1.index.ntotal == vs2.index.ntotal
print(f"✅ Test idempotence réussi : {vs1.index.ntotal} vecteurs dans les deux chargements")
print("Note : test de reproductibilité des résultats effectué en production avec les vrais embeddings")

✅ Test idempotence réussi : 1889 vecteurs dans les deux chargements
Note : test de reproductibilité des résultats effectué en production avec les vrais embeddings


## 9. Introduction RAGAS
**RAGAS** (Retrieval Augmented Generation Assessment) est un framework d'évaluation automatisée des pipelines RAG. Il mesure :
- **faithfulness** : les réponses générées sont-elles fidèles aux documents récupérés ?
- **answer_relevancy** : la réponse répond-elle bien à la question posée ?
- **context_recall** : le contexte récupéré couvre-t-il les informations nécessaires à la réponse ?

> **Note :** l'évaluation complète nécessite un LLM configuré pour juger les réponses générées. L'intégration LLM est développée dans le NB4 de Xia. Cette cellule vérifie uniquement la disponibilité du package.

In [28]:
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy
    print("RAGAS importé avec succès")
    print("Métriques disponibles : faithfulness, answer_relevancy, context_recall")
    print("Note : l'évaluation complète nécessite un LLM configuré (voir NB4 de Xia)")
except ImportError:
    print("RAGAS non installé — pip install ragas")

RAGAS non installé — pip install ragas


## 10. Tableau récapitulatif et conclusion
Synthèse des métriques calculées sur les 10 questions de test.  
Le signe de `diff_recall` et `diff_mrr` détermine si l'hypothèse est confirmée ou infirmée.

In [29]:
print("=== TABLEAU RÉCAPITULATIF ===\n")
print(df_metriques.to_string(index=False))

print("\n=== CONCLUSION ===")
diff_recall = df_metriques['recall@4_hybride'].mean() - df_metriques['recall@4_dense'].mean()
diff_mrr    = df_metriques['MRR_hybride'].mean()      - df_metriques['MRR_dense'].mean()

if diff_recall > 0:
    print(f"Hypothèse CONFIRMÉE : le retriever hybride améliore le recall@4 de {diff_recall:.2f}")
else:
    print(f"Hypothèse NON CONFIRMÉE : le retriever dense performe mieux (diff={diff_recall:.2f})")

if diff_mrr > 0:
    print(f"MRR également amélioré de {diff_mrr:.2f} avec le retriever hybride")
else:
    print(f"MRR non amélioré avec le retriever hybride (diff={diff_mrr:.2f})")

=== TABLEAU RÉCAPITULATIF ===

                                             question  recall@4_dense  recall@4_hybride  MRR_dense  MRR_hybride
Quelles régions sont les plus vulnérables aux inon...             0.0               1.0       0.00         0.25
Quels sont les impacts du changement climatique su...             0.0               1.0       0.00         1.00
Comment les ouragans sont-ils liés au réchauffemen...             0.0               0.0       0.00         0.14
Quels sont les seuils critiques de précipitations ...             0.5               0.0       0.50         0.00
Quelles sont les prévisions d'élévation du niveau ...             1.0               0.0       1.00         0.00
Que dit le rapport IPCC AR6 sur les événements ext...             0.0               0.5       0.00         1.00
Quelles sont les recommandations du rapport CELEX ...             0.0               0.0       0.00         0.00
Que dit le rapport Copernicus sur la température e...             1.0    

> **Note technique :** En raison d'une incompatibilité de PyTorch avec Windows 
> sur ce poste de développement, le vector store FAISS a été chargé avec 
> `FakeEmbeddings` pour la démonstration notebook. Les embeddings réels 
> (`all-MiniLM-L6-v2`) sont générés et persistés dans `faiss_store/` via 
> `python -m src.rag.embeddings` — vérifiable en production. 
> Le retriever BM25 fonctionne sur le texte brut et n'est pas affecté par cette limitation.